# Real-Time Object Detection on KITTI with YOLOv8
### Autonomous Driving Perception Pipeline - Model Training Notebook

This notebook fine-tunes **YOLOv8** on the **KITTI 2D Object Detection Dataset** for 3 target classes:
1. `Car` (Class 0)
2. `Pedestrian` (Class 1)
3. `Cyclist` (Class 2)

Run this notebook on Google Colab with a free GPU (**Runtime -> Change runtime type -> T4 GPU**).

## 1. Verify GPU and Install Dependencies

In [ ]:
!nvidia-smi
!pip install --upgrade ultralytics onnx onnxruntime opencv-python matplotlib

## 2. Mount Google Drive or Download KITTI Dataset
Option A: Mount Google Drive if you have uploaded KITTI `data_object_image_2.zip` and `data_object_label_2.zip`.
Option B: Download directly using `wget` or kaggle.

In [ ]:
# Option A: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Option B: Create directories
import os
os.makedirs('/content/kitti_raw', exist_ok=True)
os.makedirs('/content/datasets/kitti', exist_ok=True)
print("Directories created.")

## 3. Convert KITTI Annotations to YOLO Format (3 Classes)
KITTI format: `[left, top, right, bottom]` -> YOLO normalized: `[x_center, y_center, width, height]`

In [ ]:
import os, cv2, shutil, random
from tqdm import tqdm

CLASSES = {"Car": 0, "Pedestrian": 1, "Cyclist": 2}

def convert_and_split(label_dir, img_dir, out_root, split_ratio=0.8):
    train_img = os.path.join(out_root, "images/train")
    val_img = os.path.join(out_root, "images/val")
    train_lbl = os.path.join(out_root, "labels/train")
    val_lbl = os.path.join(out_root, "labels/val")
    for d in [train_img, val_img, train_lbl, val_lbl]:
        os.makedirs(d, exist_ok=True)
        
    label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
    random.seed(42)
    random.shuffle(label_files)
    
    split_idx = int(len(label_files) * split_ratio)
    train_files = label_files[:split_idx]
    val_files = label_files[split_idx:]
    
    def process_subset(files, target_img_dir, target_lbl_dir):
        for f in tqdm(files):
            img_name = f.replace('.txt', '.png')
            img_path = os.path.join(img_dir, img_name)
            if not os.path.exists(img_path):
                continue
            img = cv2.imread(img_path)
            if img is None: continue
            h, w = img.shape[:2]
            
            lines = []
            with open(os.path.join(label_dir, f)) as lf:
                for line in lf:
                    p = line.strip().split()
                    if not p or p[0] not in CLASSES:
                        continue
                    l, t, r, b = map(float, p[4:8])
                    l, r = max(0.0, min(float(w), l)), max(0.0, min(float(w), r))
                    t, b = max(0.0, min(float(h), t)), max(0.0, min(float(h), b))
                    if r <= l or b <= t: continue
                    xc, yc = (l + r) / 2.0 / w, (t + b) / 2.0 / h
                    bw, bh = (r - l) / w, (b - t) / h
                    lines.append(f"{CLASSES[p[0]]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
            
            # Copy image and write label
            shutil.copy2(img_path, os.path.join(target_img_dir, img_name))
            with open(os.path.join(target_lbl_dir, f), 'w') as of:
                of.write('\n'.join(lines))
                
    print("Processing train set...")
    process_subset(train_files, train_img, train_lbl)
    print("Processing val set...")
    process_subset(val_files, val_img, val_lbl)
    print("Conversion and split complete!")

# Call conversion function once data paths are set
# convert_and_split('/content/kitti_raw/label_2', '/content/kitti_raw/image_2', '/content/datasets/kitti')

## 4. Create `kitti.yaml` Dataset Descriptor

In [ ]:
yaml_content = """
path: /content/datasets/kitti
train: images/train
val: images/val

names:
  0: Car
  1: Pedestrian
  2: Cyclist
"""
with open('/content/kitti.yaml', 'w') as f:
    f.write(yaml_content.strip())
print("kitti.yaml successfully written!")

## 5. Fine-Tune YOLOv8 on KITTI

In [ ]:
from ultralytics import YOLO

# Load pretrained nano model
model = YOLO('yolov8n.pt')

# Train for 50 epochs
results = model.train(
    data='/content/kitti.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='kitti_yolo',
    name='train',
    save=True
)

## 6. Validate Model and Record Metrics
**Record these numbers for your resume!**

In [ ]:
metrics = model.val()
print("=" * 50)
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print("=" * 50)

## 7. Export Best Model to ONNX
Export for high-performance deployment with ONNX Runtime / TensorRT.

In [ ]:
onnx_path = model.export(format='onnx', imgsz=640, opset=12)
print(f"Model exported to ONNX: {onnx_path}")

## 8. Download Weights
Download `best.pt` and `best.onnx` to your computer to run in the ROS 2 pipeline.

In [ ]:
from google.colab import files
# files.download('kitti_yolo/train/weights/best.pt')
# files.download('kitti_yolo/train/weights/best.onnx')